# Práctica 3 — PLN para Inteligencia de Negocios y Toma de Decisiones Estratégicas
## "Voz del Cliente para la Estrategia de Producto y Servicio" — TecnoGadget S.A.

**Autores:** Francisco Javier Cañas González · _[nombre del compañero/a]_

**Visitemos el sitio:**  https://spacy.io/

Notebook **integrado** con las tres fases de la práctica (preprocesamiento → sentimiento, NER y temas → información para la decisión), siguiendo la estructura y las correcciones de la actividad tutorizada:

1. **Normalización `NFKC`** (preserva tildes y ñ).
2. **Negaciones contradictorias** → `handle_negations` antes de quitar stopwords en el pipeline clásico; `clean_light` para NER y modelos de sentimiento.
3. **Prints masivos** → `display()`, `value_counts()` y gráficos.
4. **Preguntas decisionales (Goldratt)**: cada resultado responde a una pregunta concreta de la gerencia, no es un "listado de 100 páginas".

| Fase | Contenido |
|---|---|
| **1** | Entorno, dataset simulado, preprocesamiento y exploración inicial |
| **2** | Sentimiento (general y por aspecto), NER y modelado de temas (LDA) |
| **3** | Preguntas decisionales, información, reporte gerencial y recomendaciones |

Archivos que genera el notebook: `Datos_TecnoGadget.csv` (dataset simulado) y `Datos_TecnoGadget_analizado.csv` (DataFrame final con sentimiento, entidades y temas).

## Fase 1 — Preparación del entorno y preprocesamiento de datos

### 1.1 Configuración del entorno

In [ ]:
!pip install -q spacy pysentimiento wordcloud
!python -m spacy download es_core_news_sm -q

In [ ]:
import json
import random
import re
import unicodedata
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
from IPython.display import Markdown, display
from matplotlib.gridspec import GridSpec
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from wordcloud import WordCloud

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

# Constantes de negocio y paleta compartida por todos los gráficos
PRODUCT_TG, PRODUCT_A, PRODUCT_B = "Auriculares TecnoGadget", "Competidor A", "Competidor B"
PRODUCT_ORDER = [PRODUCT_TG, PRODUCT_A, PRODUCT_B]
BRAND_SHORT = {PRODUCT_TG: "TecnoGadget", PRODUCT_A: "Competidor A", PRODUCT_B: "Competidor B"}
PRODUCT_COLORS = {PRODUCT_TG: "#264653", PRODUCT_A: "#e9c46a", PRODUCT_B: "#f4a261"}
SOURCE_ORDER = ["Amazon", "ForoX", "X", "ServicioCliente"]
SENT_ORDER = ["POS", "NEU", "NEG"]
SENT_COLORS = {"POS": "#2a9d8f", "NEU": "#adb5bd", "NEG": "#e76f51"}


def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")


nlp = spacy.load("es_core_news_sm")
print("spaCy:", nlp.meta["name"], nlp.meta["version"])

### 1.2 Adquisición de datos (simulada)

Por privacidad no se usan datos reales: se **simula** un dataset de ~1500 comentarios con la estructura pedida (`ID, Fuente, Producto, Puntuacion, Texto_Comentario`). El generador es reproducible (semilla fija) y está parametrizado para que el dataset:

- Cubra **4 fuentes** con estilos distintos: `Amazon` (reseñas medias), `ForoX` (largas, con preguntas), `X` (cortas, emojis, hashtags, menciones, URLs) y `ServicioCliente` (transcripciones de chat/llamada).
- Incluya **jerga, quejas, elogios, preguntas, comentarios neutros, comparaciones entre marcas, ciudades, nombres de agentes, errores (texto sin tildes, todo en minúsculas)**.
- Represente los **3 desafíos estratégicos**: características del producto (rediseño), puntos de dolor del servicio y comparación con `Competidor A` / `Competidor B`.
- **Evite sesgos**: la `Puntuacion` se deriva del sentimiento real de lo que se dice (no es aleatoria), los tres productos tienen volúmenes comparables y cada uno tiene fortalezas y debilidades propias (ver memoria de datos en el apartado 1.6).

In [ ]:
# ---------- Generador del dataset simulado (plantillas parametrizadas + semilla fija) ----------
# Perfil de percepción simulado: aspecto -> (peso de mención, probabilidad de opinión positiva)
PROFILES = {
    PRODUCT_TG: {"sonido": (1.0, .88), "cancelación de ruido": (.8, .82), "batería": (.8, .70),
                 "graves": (.4, .55), "comodidad y ajuste": (.7, .28), "conectividad": (.7, .35),
                 "micrófono": (.6, .30), "app y ecualizador": (.5, .75), "diseño y estuche": (.5, .70),
                 "precio": (.6, .40)},
    PRODUCT_A: {"sonido": (.9, .65), "cancelación de ruido": (.5, .50), "batería": (.8, .35),
                "graves": (.4, .55), "comodidad y ajuste": (.7, .80), "conectividad": (.8, .88),
                "micrófono": (.5, .60), "app y ecualizador": (.4, .50), "diseño y estuche": (.6, .35),
                "precio": (.6, .70)},
    PRODUCT_B: {"sonido": (.9, .45), "cancelación de ruido": (.4, .35), "batería": (.6, .60),
                "graves": (.5, .45), "comodidad y ajuste": (.6, .55), "conectividad": (.8, .30),
                "micrófono": (.5, .40), "app y ecualizador": (.3, .45), "diseño y estuche": (.5, .60),
                "precio": (1.0, .85)},
}
SERVICE_PROFILE = {"tiempos de espera": (1.0, .15), "garantía y devoluciones": (.8, .30),
                   "atención del agente": (.8, .55), "chatbot": (.55, .20)}
SERVICE_ISSUES = {"conectividad": .30, "micrófono": .20, "batería": .20, "comodidad y ajuste": .10,
                  "diseño y estuche": .10, "app y ecualizador": .10}
TERM = {"sonido": "sonido", "graves": "graves", "cancelación de ruido": "cancelación de ruido",
        "batería": "batería", "comodidad y ajuste": "ajuste", "conectividad": "conexión",
        "micrófono": "micrófono", "app y ecualizador": "app", "diseño y estuche": "diseño", "precio": "precio"}

BANK = {
    "sonido": {
        "pos": ["El sonido es increíble, con unos medios muy limpios", "Calidad de audio de otro nivel",
                "Suenan de lujo, el sonido es nítido y equilibrado", "El sonido es una pasada, se oye cada instrumento",
                "Un sonido brutal, me ha sorprendido para ser inalámbricos",
                "Audio cristalino, se nota la mejora respecto a mis anteriores",
                "El sonido es espectacular tanto en música como en podcasts"],
        "neg": ["El sonido es plano y sin vida", "Calidad de audio justita, esperaba bastante más",
                "El sonido se distorsiona cuando subes el volumen", "El sonido es metálico y cansa a la hora de uso",
                "Los medios están enterrados y el sonido resulta apagado", "El audio tiene un siseo de fondo molesto",
                "El sonido es mediocre, unos auriculares de 20 euros suenan igual"]},
    "graves": {
        "pos": ["Los graves tienen pegada sin embarrar el resto", "Unos bajos contundentes y controlados",
                "Los graves son potentes, ideales para electrónica", "Graves profundos y bien definidos"],
        "neg": ["Los graves se quedan cortos", "Los bajos son flojos, echo en falta pegada",
                "Los graves están muy contenidos, ni se sienten", "Los bajos retumban y lo embarran todo"]},
    "cancelación de ruido": {
        "pos": ["La cancelación de ruido es excelente, en el tren ni me entero", "La cancelación supera lo esperado",
                "Cancelación de ruido muy potente, el avión desaparece", "El ANC funciona genial en la oficina",
                "Aislamiento y cancelación de ruido de gama alta"],
        "neg": ["La cancelación de ruido es floja, se cuela todo el ruido de la calle",
                "La cancelación apenas se nota, el ANC es decorativo",
                "Cancelación de ruido irregular, mete un soplido molesto",
                "El ANC no aísla del tráfico, esperaba mucho más"]},
    "batería": {
        "pos": ["La batería dura todo el día sin despeinarse", "Autonomía brutal, llego a la semana de uso normal",
                "Carga rápida: diez minutos y tengo horas de música",
                "La batería es de lo mejor que he probado, 30 horas reales",
                "La duración de la batería está muy por encima de lo que anuncian"],
        "neg": ["La batería dura menos de lo que prometen, no llego a cuatro horas",
                "Autonomía decepcionante, a media tarde ya están muertos",
                "La batería se degrada rápido, a los tres meses ya se notaba",
                "La carga tarda una eternidad", "Batería justa: la duración real es la mitad de lo anunciado"]},
    "comodidad y ajuste": {
        "pos": ["El ajuste es cómodo incluso usándolos durante horas",
                "Muy cómodos, las almohadillas de espuma se adaptan genial",
                "Comodidad total, casi ni notas que los llevas", "Ligeros y con un ajuste perfecto para correr",
                "Ergonomía excelente, no me duelen las orejas tras horas"],
        "neg": ["El ajuste es flojo y se caen al correr", "Las almohadillas aprietan y a la hora me duelen los oídos",
                "Poco cómodos, el peso se nota en una sesión larga",
                "El ajuste es pésimo, se mueven con cualquier giro de cabeza",
                "Incómodos para llevarlos mucho rato, las almohadillas rozan"]},
    "conectividad": {
        "pos": ["El emparejamiento es rapidísimo, conectan solos al abrir", "Conexión Bluetooth estable, cero cortes",
                "La conectividad es perfecta, ni un corte en semanas",
                "Bluetooth 5 y se nota: la latencia es mínima en vídeo",
                "El emparejamiento con el móvil es instantáneo"],
        "neg": ["Se desconectan solos cada dos por tres, la conexión es un desastre",
                "Bluetooth inestable, cortes constantes al mover el móvil",
                "Problemas de emparejamiento: a veces solo conecta un auricular",
                "La latencia es horrible en vídeo y juegos", "Conectividad pésima, se pierde la señal en la calle"]},
    "micrófono": {
        "pos": ["El micrófono se oye muy bien en llamadas",
                "Las llamadas se escuchan nítidas, el micrófono filtra bien el ruido",
                "Micro de buena calidad, en videoconferencias me oyen perfecto"],
        "neg": ["El micrófono es malísimo, en las llamadas me oyen como en una lata",
                "Las llamadas con el micro son un horror, no filtra el viento",
                "Micrófono muy flojo, mis compañeros no me oyen en las reuniones",
                "El micro capta todo el ruido de fondo en las llamadas",
                "Micrófono con eco, inutilizable para trabajar"]},
    "app y ecualizador": {
        "pos": ["La app de ecualización es muy potente",
                "La aplicación es intuitiva y el ecualizador lo deja todo a tu gusto",
                "App muy completa: ecualizador, modos de ruido y actualizaciones",
                "La app funciona de maravilla y es muy rápida"],
        "neg": ["La app es un desastre, se cierra y pierde la configuración del ecualizador",
                "La aplicación es lenta y está llena de errores",
                "App obligatoria para activar funciones y encima va fatal",
                "El ecualizador de la app no guarda los perfiles"]},
    "diseño y estuche": {
        "pos": ["El diseño es elegante y los materiales se sienten premium",
                "Acabado excelente, el estuche es compacto y bonito",
                "Construcción sólida y estética muy cuidada",
                "El estuche es pequeño y la construcción transmite calidad"],
        "neg": ["El estuche se raya con facilidad", "Los materiales parecen plástico de juguete, el acabado es mejorable",
                "La construcción es frágil, la bisagra del estuche se aflojó a los dos meses",
                "El diseño es feo y voluminoso, el estuche no cabe en el bolsillo",
                "El acabado se desconcha y el estuche acumula marcas"]},
    "precio": {
        "pos": ["Muy buena relación calidad-precio", "Por lo que cuestan, es una compra acertada",
                "Por el precio que tienen, superan a otros de gama más alta", "Vale cada euro, precio muy ajustado",
                "Precio competitivo para lo que ofrecen"],
        "neg": ["Demasiado caros para lo que ofrecen, el precio no se justifica",
                "El precio es excesivo, hay opciones mejores por menos dinero", "Por el precio esperaba mucho más",
                "No compensa el precio: calidad de gama media a precio premium", "Relación calidad-precio floja"]},
}
SERVICE_BANK = {
    "tiempos de espera": {
        "pos": ["Me atendieron en menos de cinco minutos, casi sin espera",
                "Respuesta rápida, el tiempo de respuesta fue el mismo día", "Cero cola, muy ágiles"],
        "neg": ["Estuve cuarenta minutos en espera hasta que cogieron la llamada",
                "Tardaron una semana en contestar mi correo",
                "Me tuvieron en cola más de media hora y se cortó la llamada", "Llevo diez días esperando respuesta",
                "El tiempo de respuesta es desesperante, tres días para un estamos revisando"]},
    "garantía y devoluciones": {
        "pos": ["Me enviaron un reemplazo en 48 horas sin discutir",
                "La devolución fue sencilla y el reembolso llegó rápido", "Garantía sin complicaciones, cambio inmediato"],
        "neg": ["Me piden reenviar los auriculares a mi costa y la garantía tarda semanas",
                "Me negaron la devolución alegando mal uso, cosa que no es cierta",
                "El reembolso lleva un mes sin aparecer en mi cuenta",
                "Me mandaron un reemplazo que llegó igual de defectuoso", "La garantía es un laberinto de formularios"]},
    "atención del agente": {
        "pos": ["El técnico fue amable y me guió paso a paso hasta resolverlo",
                "{agent} me atendió genial y lo solucionó en la misma llamada",
                "Muy profesional el agente, explicó todo con claridad"],
        "neg": ["El agente no sabía resolver el problema y me pasó de un departamento a otro",
                "Me repitieron las mismas preguntas tres veces y no solucionaron nada",
                "Trato seco y respuestas de manual, nada resolutivo",
                "Tuve que explicar el problema a cuatro agentes distintos",
                "{agent} me atendió con desgana y no resolvió nada"]},
    "chatbot": {
        "pos": ["El chatbot me resolvió la duda al momento",
                "El asistente virtual me derivó rápido a un agente humano"],
        "neg": ["El chatbot no entendía nada y me mandaba siempre al mismo menú",
                "Imposible hablar con una persona, el bot solo repite enlaces de preguntas frecuentes",
                "El asistente virtual me dio mil vueltas sin resolver nada"]},
}
COMPARE = {"pos": ["En {t}, mejores que {o}", "Superan a {o} en {t}", "En {t} no hay color con {o}"],
           "neg": ["En {t}, peores que {o}", "{o} los supera en {t}", "En {t} me quedo con {o}"]}
Q_NEUTRAL = ["¿Qué tal van de {t}? Estoy dudando antes de comprar", "¿Alguien sabe qué tal están de {t}?",
             "¿Cómo andan de {t}? Acepto opiniones", "¿Merece la pena fijarse en {t}? Estoy comparando modelos"]
Q_COMPLAINT = ["¿Alguien más tiene fallos de {t}? Los míos van fatal",
               "¿Es normal tener problemas de {t} a las dos semanas?"]
FILLER = ["Pedido recibido en {city} en 24 horas, todavía no los he probado", "Los compré para regalar, de momento sin opinión",
          "Llegaron el martes a {city}. Los pruebo esta semana y os cuento", "Todavía en proceso de prueba, sin conclusiones"]
OPENERS = ["Tras dos semanas de uso:", "Después de un mes:", "Primeras impresiones:", "Los uso a diario para trabajar.",
           "Los compré para ir al gimnasio.", "Llevo un par de meses con ellos."]
CLOSERS = {"pos": ["Los recomiendo.", "Muy contento con la compra.", "Repetiría sin dudarlo.", "Totalmente recomendables."],
           "neg": ["No los volvería a comprar.", "Una decepción.", "Los devuelvo.", "Muy frustrante.",
                   "Mejor ir a por otra cosa."]}
SERVICE_OPENERS = ["Llamé por teléfono por un fallo:", "Escribí a la empresa por un problema con mis auriculares:",
                   "Abrí una incidencia en la web:", "Contacté tras recibir los auriculares:"]
POS_EMOJI, NEG_EMOJI = ["😍", "🔥", "👍", "💯", "😊", "🎧"], ["😡", "👎", "😤", "🙄", "😒"]
CITIES = ["Madrid", "Barcelona", "Valencia", "Sevilla", "Bilbao", "Santa Cruz de Tenerife", "Las Palmas", "Zaragoza", "Málaga"]
AGENTS = ["Laura", "Carlos", "Marta", "Javier", "Sergio", "Lucía", "Andrés", "Patricia"]
SYNONYMS = [(" muy ", [" muy ", " súper ", " bastante ", " realmente "]),
            ("increíble", ["increíble", "brutal", "espectacular"]), ("genial", ["genial", "de lujo", "estupendo"]),
            ("desastre", ["desastre", "chapuza", "horror"]), ("pésimo", ["pésimo", "malísimo", "fatal"])]


def lcfirst(s):
    return s if s.split(" ")[0] in AGENTS else s[:1].lower() + s[1:]


def weighted_sample(rnd, weights, k):
    keys, w, out = list(weights), list(weights.values()), []
    for _ in range(min(k, len(keys))):
        pick = rnd.choices(keys, weights=w, k=1)[0]
        i = keys.index(pick)
        out.append(pick); keys.pop(i); w.pop(i)
    return out


def join_clauses(items, rnd):
    """Une cláusulas; si cambia la polaridad usa conectores adversativos (pero / aunque / eso sí)."""
    text = items[0][0]
    for (clause, pol), (_, prev) in zip(items[1:], items[:-1]):
        if pol != prev:
            text += rnd.choice([", pero ", ". Eso sí, ", "; aunque "]) + lcfirst(clause)
        else:
            sep = rnd.choice([", y ", ". ", "; ", ", además "])
            text += sep + (clause if sep == ". " else lcfirst(clause))
    return text


def augment(text, rnd):
    for pat, opts in SYNONYMS:
        if pat in text and rnd.random() < .5:
            text = text.replace(pat, rnd.choice(opts), 1)
    return text


def make_review(rnd, product, source):
    prof = PROFILES[product]
    weights = {a: w for a, (w, _) in prof.items()}
    r, q_prob = rnd.random(), (.08 if source == "ForoX" else .02)
    if r < q_prob:                                           # preguntas
        a = rnd.choice(list(prof)); t = TERM[a]
        if prof[a][1] < .5 and rnd.random() < .5:
            return rnd.choice(Q_COMPLAINT).format(t=t), 2, 0.0
        return rnd.choice(Q_NEUTRAL).format(t=t), 3, 0.0
    if r < q_prob + .04:                                     # comentarios neutros
        return rnd.choice(FILLER).format(city=rnd.choice(CITIES)), 3, 0.0
    k = {"X": rnd.choices([1, 2], [.7, .3])[0], "Amazon": rnd.choices([1, 2, 3], [.25, .5, .25])[0],
         "ForoX": rnd.choices([2, 3], [.5, .5])[0]}[source]
    items, signs = [], []
    others = [p for p in BRAND_SHORT if p != product]
    for a in weighted_sample(rnd, weights, k):
        pol = "pos" if rnd.random() < prof[a][1] else "neg"
        if rnd.random() < .08:                               # comparación con otra marca
            clause = rnd.choice(COMPARE[pol]).format(t=TERM[a], o=BRAND_SHORT[rnd.choice(others)])
        else:
            clause = rnd.choice(BANK[a][pol])
        items.append((clause, pol)); signs.append(1 if pol == "pos" else -1)
    body, m, parts = join_clauses(items, rnd), float(np.mean(signs)), []
    if source != "X" and rnd.random() < .3:
        parts.append(rnd.choice(OPENERS))
    if source == "Amazon" and rnd.random() < .06:
        body += ". Llegaron a " + rnd.choice(CITIES) + " en dos días"
    parts.append(body)
    if source != "X" and abs(m) > .3 and rnd.random() < .4:
        parts.append(rnd.choice(CLOSERS["pos" if m > 0 else "neg"]))
    text = " ".join(p if p.endswith((".", ":")) else p + "." for p in parts)
    rating = int(np.clip(round(3 + 1.6 * m + rnd.gauss(0, .55)), 1, 5))
    return text, rating, m


def make_service(rnd):
    parts, signs = [rnd.choice(SERVICE_OPENERS)], []
    if rnd.random() < .75:                                   # defecto que motiva el contacto
        issue = weighted_sample(rnd, SERVICE_ISSUES, 1)[0]
        parts.append(rnd.choice(BANK[issue]["neg"]) + ".")
        signs.append(-.5)
    clauses = []
    for a in weighted_sample(rnd, {a: w for a, (w, _) in SERVICE_PROFILE.items()}, rnd.choices([1, 2], [.55, .45])[0]):
        pol = "pos" if rnd.random() < SERVICE_PROFILE[a][1] else "neg"
        clauses.append((rnd.choice(SERVICE_BANK[a][pol]).format(agent=rnd.choice(AGENTS)), pol))
        signs.append(1 if pol == "pos" else -1)
    parts.append(join_clauses(clauses, rnd) + ".")
    m = float(np.mean(signs))
    rating = int(np.clip(round(2.7 + 1.5 * m + rnd.gauss(0, .5)), 1, 5))
    return " ".join(parts), rating, m


def decorate(rnd, text, source, m):
    """Ruido propio de cada canal: emojis, hashtags, URLs, menciones, minúsculas y textos sin tildes."""
    text = augment(" " + text + " ", rnd).strip()
    if source == "X":
        if rnd.random() < .5 and abs(m) > .2:
            text += " " + rnd.choice(POS_EMOJI if m > 0 else NEG_EMOJI)
        if rnd.random() < .3:
            text += " " + rnd.choice(["#auriculares", "#tech", "#reseña", "#audio"])
        if rnd.random() < .12:
            text += " https://t.co/" + "".join(rnd.choices("abcdefghijklmnopqrstuvwxyz0123456789", k=8))
        if rnd.random() < .15:
            text = "@TecnoGadget_ES " + text
        if rnd.random() < .35:
            text = text.lower()
    if rnd.random() < .04:
        text = strip_accents(text)
    return text


def generate_dataset(n=1500, seed=42):
    rnd = random.Random(seed)
    n_service = int(n * .20)
    rows = []
    for _ in range(n - n_service):
        source = rnd.choices(["Amazon", "ForoX", "X"], [.50, .28, .22])[0]
        product = rnd.choices(PRODUCT_ORDER, [.52, .24, .24])[0]
        text, rating, m = make_review(rnd, product, source)
        rows.append((source, product, rating, decorate(rnd, text, source, m)))
    for _ in range(n_service):
        text, rating, m = make_service(rnd)
        rows.append(("ServicioCliente", PRODUCT_TG, rating, decorate(rnd, text, "ServicioCliente", m)))
    rnd.shuffle(rows)
    out = pd.DataFrame(rows, columns=["Fuente", "Producto", "Puntuacion", "Texto_Comentario"])
    out.insert(0, "ID", range(1, len(out) + 1))
    return out

### 1.3 Carga y validación del dataset
Si `Datos_TecnoGadget.csv` ya existe (por ejemplo, subido a Colab) se carga tal cual; si no, se genera y se guarda.

In [ ]:
DATA_PATH = Path("Datos_TecnoGadget.csv")
if not DATA_PATH.exists():
    generate_dataset(n=1500, seed=SEED).to_csv(DATA_PATH, index=False, encoding="utf-8")

df = pd.read_csv(DATA_PATH)
print(f"Filas: {len(df)} | Columnas: {df.columns.tolist()}")
display(df.groupby("Fuente").sample(2, random_state=SEED)[["ID", "Fuente", "Producto", "Puntuacion", "Texto_Comentario"]])

In [ ]:
display(pd.DataFrame({"nulos": df.isna().sum(), "tipo": df.dtypes.astype(str)}).T)
print(f"Textos duplicados: {df['Texto_Comentario'].duplicated().sum()} "
      f"({df['Texto_Comentario'].duplicated().mean():.1%})")

display(pd.crosstab(df["Producto"], df["Fuente"], margins=True).reindex(columns=SOURCE_ORDER + ["All"]))
# Control de sesgo: el reparto de puntuaciones se compara solo en reseñas (el servicio solo atiende a TecnoGadget)
display(df[df["Fuente"] != "ServicioCliente"].groupby("Producto")["Puntuacion"].agg(["count", "mean", "std"]).round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x="Puntuacion", hue="Producto", hue_order=PRODUCT_ORDER, palette=PRODUCT_COLORS, ax=axes[0])
axes[0].set_title("Distribución de puntuaciones por producto")
sns.countplot(data=df, y="Fuente", order=SOURCE_ORDER, hue="Producto", hue_order=PRODUCT_ORDER,
              palette=PRODUCT_COLORS, ax=axes[1])
axes[1].set_title("Comentarios por fuente y producto")
plt.tight_layout(); plt.show()

### 1.4 Preprocesamiento del texto

Pipeline clásico (bolsa de palabras, nubes y LDA):

`NFKC → emojis → URLs/menciones/#hashtags → restaurar tildes de la taxonomía → minúsculas → tokenización → lematización → negaciones → stopwords`

- **Normalización**: `NFKC` preserva tildes, ñ y ü. Se eliminan URLs, menciones (`@usuario`), caracteres especiales y números.
- **Emojis (opcional avanzado)**: los emojis con carga emocional se traducen a una palabra (`😍 → encanta`, `👎 → mal`); el resto se descarta.
- **Negaciones**: `handle_negations` une la negación con la siguiente palabra con contenido (ventana de 3 tokens, sin cruzar puntuación) **antes** de quitar stopwords. Así `no es bueno → no_bueno` y no se confunde con `bueno`.
- **Stopwords**: lista de spaCy **menos** las palabras con carga de opinión (`bien, mal, mejor, peor, poco...`) más stopwords de dominio (`auricular, tecnogadget, competidor...`).
- Para **NER y modelos de sentimiento** se usa `clean_light` (texto casi natural), no el pipeline clásico.

#### Taxonomía de aspectos
La misma taxonomía sirve para el NER personalizado (2.2), el sentimiento por aspecto (2.4) y el etiquetado automático de temas (2.3).

In [ ]:
PRODUCT_ASPECTS = {
    "sonido": ["sonido", "audio", "agudos", "medios", "calidad de sonido"],
    "graves": ["graves", "bajos"],
    "cancelación de ruido": ["cancelación de ruido", "cancelación", "anc", "aislamiento"],
    "batería": ["batería", "autonomía", "carga rápida", "carga", "duración"],
    "comodidad y ajuste": ["ajuste", "comodidad", "cómodos", "cómodo", "incómodos", "almohadillas", "peso", "ergonomía"],
    "conectividad": ["bluetooth", "conexión", "conectividad", "emparejamiento", "latencia", "cortes", "desconexiones"],
    "micrófono": ["micrófono", "micro", "micros"],
    "app y ecualizador": ["app", "aplicación", "ecualizador", "ecualización"],
    "diseño y estuche": ["diseño", "estuche", "acabado", "materiales", "construcción", "estética"],
    "precio": ["precio", "calidad-precio", "euros", "euro", "dinero", "caros", "cuestan", "cuesta"],
}
SERVICE_ASPECTS = {
    "tiempos de espera": ["espera", "cola", "tiempo de respuesta", "tiempos de espera", "tardaron", "demora"],
    "garantía y devoluciones": ["garantía", "devolución", "devoluciones", "reembolso", "reemplazo"],
    "atención del agente": ["agente", "agentes", "técnico", "atención", "trato", "soporte"],
    "chatbot": ["chatbot", "bot", "asistente virtual"],
}
ASPECT_TAXONOMY = {**PRODUCT_ASPECTS, **SERVICE_ASPECTS}
ASPECT_KIND = {**{a: "CARACTERISTICA" for a in PRODUCT_ASPECTS}, **{a: "SERVICIO" for a in SERVICE_ASPECTS}}

# Agrupación de aspectos en las 3 dimensiones del análisis competitivo (Pregunta 3)
DIMENSIONES = {
    "Calidad del producto": ["sonido", "graves", "diseño y estuche", "comodidad y ajuste"],
    "Funcionalidades": ["cancelación de ruido", "batería", "conectividad", "micrófono", "app y ecualizador"],
    "Relación calidad-precio": ["precio"],
}

#### Utilidades compartidas

In [ ]:
# variante sin tilde -> forma canónica (comentarios escritos sin acentos: "bateria" -> "batería")
CANON = {strip_accents(t): t for terms in ASPECT_TAXONOMY.values() for t in terms if strip_accents(t) != t}
CANON_RE = re.compile(r"\b(" + "|".join(sorted(map(re.escape, CANON), key=len, reverse=True)) + r")\b", re.I)

EMOJI_MAP = {"😍": " encanta ", "🔥": " genial ", "👍": " bien ", "💯": " perfecto ", "😊": " contento ",
             "😡": " enfadado ", "👎": " mal ", "😤": " harto ", "🙄": " decepcionante ", "😒": " decepcionado "}
EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF\uFE0F]")
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")

NEGATION_WORDS = {"no", "nunca", "jamás", "jamas", "ni", "tampoco", "ningún", "ninguno", "ninguna", "nadie", "nada", "sin"}
SENTIMENT_KEEP = {"bien", "bueno", "buena", "buenos", "buenas", "buen", "mal", "mejor", "mejores", "peor", "peores",
                  "gran", "grande", "fatal", "mucho", "poco", "demasiado", "más", "menos"}
DOMAIN_STOPWORDS = {"auricular", "auriculares", "tecnogadget", "competidor", "tecnogadget_es", "producto", "ser", "estar",
                    "tener", "hacer", "ir", "poder", "haber", "dar", "ver", "llegar", "pasar",
                    "llamer", "abrí", "contacté", "escribí", "él",
                    # palabras muy frecuentes pero poco informativas en nubes y temas
                    "hora", "mes", "superar", "notar", "oir", "día", "vez", "cosa", "compra", "comprar", "recibir",
                    "probar", "trabajar", "quedar", "parecer", "llevar", "decir", "sentir", "pedir", "esperar",
                    "encontrar", "repetir", "conseguir"}
# términos de la taxonomía: se conserva la forma tal cual (el lematizador pequeño los deforma: "micrófono" -> "micrófonir")
TAXONOMY_WORDS = {t for terms in ASPECT_TAXONOMY.values() for t in terms if " " not in t}
STOPWORDS = (set(nlp.Defaults.stop_words) - SENTIMENT_KEEP) | DOMAIN_STOPWORDS


def normalize_text(text):
    """NFKC preserva tildes, ñ y ü."""
    if not isinstance(text, str):
        return ""
    return unicodedata.normalize("NFKC", text)


def replace_emojis(text):
    for emoji, word in EMOJI_MAP.items():
        text = text.replace(emoji, word)
    return EMOJI_RE.sub(" ", text)


def clean_social(text):
    """Quita URLs y @menciones; deja el hashtag como palabra."""
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    return re.sub(r"#(\w+)", r"\1", text)


def restore_accents(text):
    return CANON_RE.sub(lambda m: CANON[m.group(0).lower()], text)


def clean_light(text):
    """Limpieza mínima (NER y transformers): NFKC, sin URLs/menciones/emojis, espacios colapsados."""
    if not isinstance(text, str) or not text.strip():
        return ""
    text = EMOJI_RE.sub(" ", clean_social(normalize_text(text)))
    return re.sub(r"\s+", " ", text).strip()


def is_stop(tok):
    return tok["lower"] in STOPWORDS or tok["lemma"] in STOPWORDS


def handle_negations(tokens, window=3):
    """Une negación + siguiente palabra con contenido. Debe aplicarse ANTES de quitar stopwords."""
    out, i = [], 0
    while i < len(tokens):
        tok = tokens[i]
        if tok["lower"] in NEGATION_WORDS:
            j, target = i + 1, None
            while j < len(tokens) and j <= i + window and not tokens[j]["punct"]:
                if not is_stop(tokens[j]) and not tokens[j]["num"]:
                    target = j
                    break
                j += 1
            if target is not None:                      # par negado: "no_funcionar"
                out.append(f"{tok['lower']}_{tokens[target]['lemma']}")
                i = target + 1
            else:                                       # negación sin objetivo: se descarta
                i += 1
            continue
        out.append(tok)
        i += 1
    return out


def preprocess_corpus(texts):
    """Pipeline clásico: NFKC, emojis, limpieza, lematización, negaciones ANTES de stopwords."""
    prepared = []
    for t in texts:
        t = restore_accents(clean_social(replace_emojis(normalize_text(t)))).lower()
        prepared.append(re.sub(r"[^\w\sáéíóúüñ.,;:!?¿¡-]", " ", t))
    results = []
    for doc in nlp.pipe(prepared, batch_size=64, disable=["parser", "ner"]):
        toks = [{"lower": t.lower_,
                 "lemma": t.lower_ if t.lower_ in TAXONOMY_WORDS else (t.lemma_.lower().split() or [t.lower_])[0],
                 "punct": t.is_punct, "num": t.like_num} for t in doc if not t.is_space]
        final = []
        for t in handle_negations(toks):                # ANTES de stopwords
            if isinstance(t, str):                      # par negado: conservar
                final.append(t)
            elif not (t["punct"] or t["num"] or is_stop(t) or len(t["lemma"]) < 2):
                final.append(t["lemma"])
        results.append(final)
    return results


def tokenize_raw_text(text):
    return [t.text for t in nlp.make_doc(normalize_text(text)) if not t.is_space]

In [ ]:
# Demostración del manejo de emojis y negaciones
ejemplos = ["No es bueno, no funciona", "No está mal el sonido", "Sin cortes de conexión", "Nada recomendable, no lo compraría"]
display(pd.DataFrame({"texto": ejemplos, "tokens": preprocess_corpus(ejemplos)}))

con_emoji = df[df["Texto_Comentario"].str.contains(EMOJI_RE)].head(3)["Texto_Comentario"]
display(pd.DataFrame({"original": con_emoji, "tras_emojis": con_emoji.apply(lambda t: replace_emojis(normalize_text(t)))}))

In [ ]:
df["texto_limpio"] = df["Texto_Comentario"].apply(clean_light)
df["processed_tokens"] = preprocess_corpus(df["Texto_Comentario"])
display(df[["ID", "Texto_Comentario", "processed_tokens"]].sample(5, random_state=SEED))

### 1.5 Exploración inicial

In [ ]:
df["raw_tokens"] = df["Texto_Comentario"].apply(tokenize_raw_text)
df["raw_length"] = df["raw_tokens"].apply(len)
df["processed_length"] = df["processed_tokens"].apply(len)

raw_freq = Counter(t.lower() for toks in df["raw_tokens"] for t in toks)
proc_freq = Counter(t for toks in df["processed_tokens"] for t in toks)

display(pd.DataFrame({
    "antes": [len(df), df["raw_length"].mean().round(2), len(raw_freq)],
    "después": [len(df), df["processed_length"].mean().round(2), len(proc_freq)],
}, index=["Comentarios", "Longitud media (tokens)", "Vocabulario único"]))

top = pd.concat({"antes": pd.Series(dict(raw_freq.most_common(10))).rename("frecuencia").reset_index(),
                 "después": pd.Series(dict(proc_freq.most_common(10))).rename("frecuencia").reset_index()}, axis=1)
display(top)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=df, x="Fuente", y="raw_length", order=SOURCE_ORDER, hue="Fuente", hue_order=SOURCE_ORDER,
            palette="Set2", legend=False, ax=axes[0])
axes[0].set_title("Longitud del comentario por fuente (tokens, antes)")
sns.histplot(df["processed_length"], bins=25, color="#264653", ax=axes[1])
axes[1].set_title("Longitud tras el preprocesamiento (tokens)")
plt.tight_layout(); plt.show()

In [ ]:
def plot_wordclouds(groups, title, ncols, min_freq=3):
    """Una nube de palabras por grupo {nombre: lista de tokens}."""
    nrows = int(np.ceil(len(groups) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.2 * ncols, 3.6 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, (name, toks) in zip(axes, groups.items()):
        freq = {w: f for w, f in Counter(toks).items() if f >= min_freq}
        wc = WordCloud(width=700, height=350, background_color="white", colormap="viridis",
                       max_words=60, collocations=False).generate_from_frequencies(freq)
        ax.imshow(wc, interpolation="bilinear")
        ax.set_title(name)
        ax.axis("off")
    for ax in axes[len(groups):]:
        ax.axis("off")
    fig.suptitle(title, fontsize=14, y=1.0)
    plt.tight_layout(); plt.show()


tokens_by = lambda col, val: [t for toks in df.loc[df[col] == val, "processed_tokens"] for t in toks]
plot_wordclouds({p: tokens_by("Producto", p) for p in PRODUCT_ORDER}, "Nube de palabras por producto", ncols=3)
plot_wordclouds({s: tokens_by("Fuente", s) for s in SOURCE_ORDER}, "Nube de palabras por fuente", ncols=2)

### 1.6 Documento de descripción de datos (memoria)

**Estructura del dataset (`Datos_TecnoGadget.csv`)**

| Columna | Descripción |
|---|---|
| `ID` | Identificador numérico del comentario |
| `Fuente` | `Amazon`, `ForoX`, `X` o `ServicioCliente` (transcripciones de chat/llamada) |
| `Producto` | `Auriculares TecnoGadget`, `Competidor A` o `Competidor B` |
| `Puntuacion` | 1–5 estrellas (en `ServicioCliente` se interpreta como satisfacción con la atención) |
| `Texto_Comentario` | Texto libre en español, con jerga, errores y ruido propio de cada canal |

**Decisiones de diseño del dataset**
- **Volumen y balance**: 1500 comentarios; ~52 % sobre TecnoGadget y ~24 % sobre cada competidor. El 20 % son transcripciones de `ServicioCliente`, que por definición solo atienden a TecnoGadget (no existe servicio de los competidores en la muestra). Por eso las comparaciones entre productos (Pregunta 3) usan **solo reseñas** (`Amazon`, `ForoX`, `X`).
- **Coherencia puntuación–texto**: la `Puntuacion` se calcula a partir de la polaridad de lo que se escribe más ruido gaussiano; así no hay estrellas aleatorias, pero tampoco una correlación perfecta.
- **Perfiles de percepción** (hipótesis de simulación, sin revelar a los modelos): cada producto tiene fortalezas y debilidades propias (p. ej. TecnoGadget destaca en sonido y cancelación pero flojea en ajuste y conectividad; Competidor A es el más estable y cómodo; Competidor B es el más barato). Sirve para **validar** que el análisis recupera lo que sabemos que está en los datos.
- **Variedad lingüística**: elogios, quejas, preguntas (≈2–8 %), comentarios neutros (≈4 %), comparaciones entre marcas (≈8 % de las menciones), ciudades y nombres de agentes (para NER), texto sin tildes (4 %), minúsculas, emojis, hashtags, menciones y URLs en `X`.

**Decisiones de preprocesamiento**
- `NFKC` en lugar de ASCII: se conservan tildes y ñ; los textos sin tilde se restauran solo para los términos de la taxonomía.
- Dos niveles de limpieza: **`preprocess_corpus`** (lematizado, sin stopwords, para frecuencias, nubes y LDA) y **`clean_light`** (casi natural, para NER y modelos de sentimiento, que necesitan el contexto completo).
- **Negaciones antes de stopwords** (`no_bueno`, `sin_corte`) y **stopwords sin palabras de opinión** (`bien, mal, mejor, poco...`).
- Emojis: se traducen a una palabra de sentimiento (solo los que aportan opinión).

## Fase 2 — Análisis de sentimiento, entidades y temas

### 2.1 Análisis de sentimiento (polaridad general)

Se usa **`pysentimiento`** (modelo RoBERTa específico para español) sobre `clean_light`, no sobre los tokens lematizados: los transformers necesitan el texto natural. La **polaridad** es `P(POS) − P(NEG)` (de −1 a +1) y `sentiment_score` es la confianza de la clase elegida.

> En Colab conviene activar GPU (*Entorno de ejecución → Cambiar tipo → T4*). Con CPU tarda unos minutos más.

In [ ]:
from pysentimiento import create_analyzer

analyzer = create_analyzer(task="sentiment", lang="es")


def predict_sentiment(texts, batch_size=64):
    """Devuelve (etiqueta, P(POS), P(NEU), P(NEG)) por texto, en lotes."""
    out = []
    for i in range(0, len(texts), batch_size):
        for r in analyzer.predict(texts[i:i + batch_size]):
            out.append((r.output.upper(), r.probas["POS"], r.probas["NEU"], r.probas["NEG"]))
    return out


sent = predict_sentiment(df["texto_limpio"].tolist())
df["sentiment_label"] = [s[0] for s in sent]
df["sentiment_score"] = [max(s[1:]) for s in sent]
df["polarity"] = [s[1] - s[3] for s in sent]

display(df[["ID", "Texto_Comentario", "sentiment_label", "sentiment_score", "polarity"]].head(8))
display(df["sentiment_label"].value_counts().reindex(SENT_ORDER).to_frame("comentarios"))

In [ ]:
# Distribución del sentimiento por producto, por fuente y por puntuación de estrellas
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
for ax, (col, order) in zip(axes, [("Producto", PRODUCT_ORDER), ("Fuente", SOURCE_ORDER), ("Puntuacion", [1, 2, 3, 4, 5])]):
    tab = pd.crosstab(df[col], df["sentiment_label"], normalize="index").reindex(index=order, columns=SENT_ORDER, fill_value=0) * 100
    tab.plot(kind="barh", stacked=True, color=[SENT_COLORS[c] for c in SENT_ORDER], ax=ax, width=.75)
    ax.set_title(f"Sentimiento por {col.lower()} (%)"); ax.set_xlabel(""); ax.set_ylabel("")
    ax.legend(title="", loc="lower center", bbox_to_anchor=(.5, -.28), ncol=3, frameon=False)
    ax.invert_yaxis()
plt.tight_layout(); plt.show()

**Validación crítica**: comparamos el sentimiento del modelo con las estrellas (≥4 → POS, 3 → NEU, ≤2 → NEG). No son la verdad absoluta (hay ruido en las puntuaciones y los comentarios mixtos), pero un acuerdo bajo indicaría un problema.

In [ ]:
star_label = df["Puntuacion"].map(lambda r: "POS" if r >= 4 else "NEG" if r <= 2 else "NEU")
rep = pd.DataFrame(classification_report(star_label, df["sentiment_label"], labels=SENT_ORDER,
                                         output_dict=True, zero_division=0)).T.round(2)
display(rep)

cm = pd.crosstab(star_label, df["sentiment_label"]).reindex(index=SENT_ORDER, columns=SENT_ORDER, fill_value=0)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Estrellas (filas) vs. sentimiento del modelo (columnas)")
sns.barplot(data=df, x="Puntuacion", y="polarity", color="#264653", ax=axes[1])
axes[1].set_title("Polaridad media por puntuación")
plt.tight_layout(); plt.show()

### 2.2 Extracción de entidades nombradas (NER)

El NER estándar de `es_core_news_sm` reconoce personas, lugares y organizaciones, pero **no conoce nuestro dominio** (características de producto, marcas, motivos de servicio). Por eso se añade un **`EntityRuler`** antes del modelo estadístico con la taxonomía de aspectos:

| Etiqueta | Origen | Ejemplos |
|---|---|---|
| `CARACTERISTICA` | EntityRuler (taxonomía) | batería, cancelación de ruido, ajuste |
| `SERVICIO` | EntityRuler (taxonomía) | reembolso, chatbot, tiempo de respuesta |
| `MARCA` | EntityRuler | TecnoGadget, Competidor A / B |
| `PER`, `LOC`, `ORG` | Modelo spaCy | Laura, Tenerife, Madrid |

Cada entidad de dominio lleva un `ent_id` con su **aspecto canónico** (`"emparejamiento"` y `"conexión"` → `conectividad`), que reutilizaremos en 2.4.

In [ ]:
ruler = nlp.add_pipe("entity_ruler", before="ner", config={"overwrite_ents": True})

patterns = []
for aspect, terms in ASPECT_TAXONOMY.items():
    for term in set(terms) | {strip_accents(t) for t in terms}:          # con y sin tilde
        patterns.append({"label": ASPECT_KIND[aspect], "id": aspect,
                         "pattern": [{"LOWER": tok.lower_} for tok in nlp.make_doc(term)]})
BRAND_PATTERNS = {"TecnoGadget": [["tecnogadget"]], "Competidor A": [["competidor", "a"], ["competidor", "a."]],
                  "Competidor B": [["competidor", "b"], ["competidor", "b."]]}
for brand, variants in BRAND_PATTERNS.items():
    for toks in variants:
        patterns.append({"label": "MARCA", "id": brand, "pattern": [{"LOWER": t} for t in toks]})
ruler.add_patterns(patterns)
print("Pipeline:", nlp.pipe_names)

In [ ]:
SENT_BOUNDARY = {".", ":", ";", "!", "?", "¿", "¡"}


def valid_entity(ent):
    """Capa de validación: el modelo pequeño etiqueta como PER/LOC verbos o frases al inicio de oración.
    Las entidades de dominio se aceptan siempre; las del modelo solo si empiezan en mayúscula, no abren oración y no son stopwords."""
    if ent.label_ in ("CARACTERISTICA", "SERVICIO", "MARCA"):
        return True
    if not ent.text[:1].isupper():
        return False
    if ent.start == 0 or ent.doc[ent.start - 1].text in SENT_BOUNDARY:
        return False
    return ent.text.lower() not in STOPWORDS


docs_ner = list(nlp.pipe(df["texto_limpio"], batch_size=64,
                         disable=["parser", "lemmatizer", "morphologizer", "attribute_ruler"]))


def extract_entities(doc):
    return [(e.ent_id_ if e.label_ in ("MARCA",) else e.text, e.label_) for e in doc.ents if valid_entity(e)]


df["entidades"] = [extract_entities(d) for d in docs_ner]

raw_counts = Counter(e.label_ for d in docs_ner for e in d.ents)
valid_counts = Counter(l for ents in df["entidades"] for _, l in ents)
display(pd.DataFrame({"detectadas (modelo)": raw_counts, "tras validación": valid_counts}).fillna(0).astype(int)
        .sort_values("detectadas (modelo)", ascending=False))
display(df[["ID", "texto_limpio", "entidades"]].loc[df["entidades"].str.len() > 2].head(5))

In [ ]:
ents_long = pd.DataFrame([(row.ID, row.Producto, row.Fuente, name, label)
                          for row in df.itertuples() for name, label in row.entidades],
                         columns=["ID", "Producto", "Fuente", "entidad", "etiqueta"])
ents_long["entidad"] = np.where(ents_long["etiqueta"].isin(["CARACTERISTICA", "SERVICIO"]),
                                ents_long["entidad"].str.lower(), ents_long["entidad"])

fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
carac = (ents_long[ents_long["etiqueta"] == "CARACTERISTICA"]["entidad"].str.lower().value_counts().head(12))
carac.sort_values().plot(kind="barh", color="#264653", ax=axes[0]); axes[0].set_title("Características más nombradas (términos)")
serv = ents_long[ents_long["etiqueta"] == "SERVICIO"]["entidad"].str.lower().value_counts().head(10)
serv.sort_values().plot(kind="barh", color="#e76f51", ax=axes[1]); axes[1].set_title("Términos de servicio más nombrados")
marca = pd.crosstab(ents_long.loc[ents_long["etiqueta"] == "MARCA", "Producto"], ents_long.loc[ents_long["etiqueta"] == "MARCA", "entidad"])
marca.reindex(PRODUCT_ORDER).fillna(0).plot(kind="bar", stacked=True, ax=axes[2], colormap="Set2", rot=15)
axes[2].set_title("Marca mencionada (columnas) según el producto reseñado"); axes[2].set_xlabel("")
plt.tight_layout(); plt.show()

display(ents_long[ents_long["etiqueta"].isin(["PER", "LOC"])].groupby("etiqueta")["entidad"]
        .agg(lambda s: ", ".join(f"{k} ({v})" for k, v in s.value_counts().head(8).items())).to_frame("más frecuentes"))

### 2.3 Modelado de temas (LDA)

LDA sobre los tokens lematizados (con negaciones) para descubrir **temas recurrentes**. Para elegir el número de temas (5–10) se compara la **perplejidad en un conjunto de validación** y la **coherencia UMass** (cuanto más cercana a 0, mejor); como ambas métricas son orientativas (y aquí varían poco entre 6 y 10 temas), la decisión final se toma por **interpretabilidad**.

In [ ]:
docs_lda = df["processed_tokens"].apply(" ".join)
vectorizer = CountVectorizer(min_df=5, max_df=0.5, token_pattern=r"(?u)\b\w{2,}\b")
X = vectorizer.fit_transform(docs_lda)
vocab = np.array(vectorizer.get_feature_names_out())
print(f"Matriz documento-término: {X.shape[0]} documentos × {X.shape[1]} términos")


def umass_coherence(components, X, n_top=10):
    """Coherencia UMass media de los temas (co-ocurrencia por documento)."""
    Xb = (X > 0).astype(int)
    co, d = (Xb.T @ Xb).toarray(), np.asarray(Xb.sum(axis=0)).ravel()
    scores = []
    for comp in components:
        top = comp.argsort()[::-1][:n_top]
        pairs = [np.log((co[top[i], top[j]] + 1) / d[top[j]]) for i in range(1, n_top) for j in range(i)]
        scores.append(np.mean(pairs))
    return float(np.mean(scores))


X_train, X_val = train_test_split(X, test_size=0.2, random_state=SEED)
rows = []
for k in range(5, 11):
    m = LatentDirichletAllocation(n_components=k, learning_method="batch", max_iter=40, random_state=SEED,
                                  doc_topic_prior=0.1, topic_word_prior=0.05).fit(X_train)
    rows.append({"temas": k, "perplejidad (val.)": m.perplexity(X_val), "coherencia UMass": umass_coherence(m.components_, X_train)})
k_table = pd.DataFrame(rows).set_index("temas")
display(k_table.round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
k_table["perplejidad (val.)"].plot(marker="o", ax=axes[0], color="#264653", title="Perplejidad (menor = mejor)")
k_table["coherencia UMass"].plot(marker="o", ax=axes[1], color="#e76f51", title="Coherencia UMass (mayor = mejor)")
plt.tight_layout(); plt.show()

In [ ]:
# Las métricas son casi planas entre 6 y 10 temas -> se elige 8 por interpretabilidad (cubre los aspectos principales + servicio)
N_TOPICS = 8
print("Número de temas elegido:", N_TOPICS)

# priores dispersos (alpha, eta bajos): textos cortos con 1-3 temas por comentario
lda = LatentDirichletAllocation(n_components=N_TOPICS, learning_method="batch", max_iter=80, random_state=SEED,
                                doc_topic_prior=0.1, topic_word_prior=0.05).fit(X)
doc_topic = lda.transform(X)
topic_word = lda.components_ / lda.components_.sum(axis=1, keepdims=True)

# Etiqueta automática: aspectos de la taxonomía con más peso en el tema (se compara con las palabras clave)
ASPECT_LEMMAS = {a: {t.lemma_.lower() for t in nlp(" ".join(terms)) if t.is_alpha} for a, terms in ASPECT_TAXONOMY.items()}
vocab_idx = {w: i for i, w in enumerate(vocab)}


def label_topic(k):
    score = {a: sum(topic_word[k, vocab_idx[w]] for w in lemmas if w in vocab_idx) for a, lemmas in ASPECT_LEMMAS.items()}
    best = sorted(score.items(), key=lambda kv: kv[1], reverse=True)[:2]
    names = [a for a, s in best if s > 0.02]
    return f"T{k + 1} · " + (" / ".join(names) if names else "valoración general")


topic_labels = {k: label_topic(k) for k in range(N_TOPICS)}
top_words = {topic_labels[k]: ", ".join(vocab[topic_word[k].argsort()[::-1][:10]]) for k in range(N_TOPICS)}
display(pd.Series(top_words, name="palabras clave (top 10)").to_frame())

In [ ]:
df["topic_id"] = doc_topic.argmax(axis=1)
df["topic_label"] = df["topic_id"].map(topic_labels)
df["topic_prob"] = doc_topic.max(axis=1)

# Lectura de negocio: peso del tema y sentimiento dominante de sus comentarios
temas = (df.groupby("topic_label").agg(comentarios=("ID", "size"), pct_neg=("sentiment_label", lambda s: (s == "NEG").mean() * 100),
                                       pct_pos=("sentiment_label", lambda s: (s == "POS").mean() * 100),
                                       polaridad=("polarity", "mean")).round(1))
temas["lectura"] = np.where(temas["pct_neg"] >= 50, "Punto débil", np.where(temas["pct_pos"] >= 50, "Fortaleza", "Mixto"))
display(temas.sort_values("comentarios", ascending=False))

fig, axes = plt.subplots(2, int(np.ceil(N_TOPICS / 2)), figsize=(4.2 * int(np.ceil(N_TOPICS / 2)), 6.5))
for k, ax in zip(range(N_TOPICS), axes.ravel()):
    idx = topic_word[k].argsort()[::-1][:8]
    ax.barh(vocab[idx][::-1], topic_word[k][idx][::-1], color="#2a9d8f")
    ax.set_title(topic_labels[k], fontsize=10)
for ax in axes.ravel()[N_TOPICS:]:
    ax.axis("off")
plt.suptitle("Palabras más representativas de cada tema", y=1.01); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.heatmap(pd.crosstab(df["Producto"], df["topic_label"], normalize="index").reindex(PRODUCT_ORDER) * 100, annot=True, fmt=".0f",
            cmap="YlGnBu", cbar=False, ax=axes[0]); axes[0].set_title("Peso de cada tema por producto (%)")
sns.heatmap(pd.crosstab(df["Fuente"], df["topic_label"], normalize="index").reindex(SOURCE_ORDER) * 100, annot=True, fmt=".0f",
            cmap="YlGnBu", cbar=False, ax=axes[1]); axes[1].set_title("Peso de cada tema por fuente (%)")
for ax in axes:
    ax.set_xlabel(""); ax.set_ylabel(""); ax.tick_params(axis="x", rotation=40)
plt.tight_layout(); plt.show()

### 2.4 Sentimiento por aspecto (opcional avanzado)

Se desarrolla tras el NER porque **reutiliza las entidades `CARACTERISTICA` / `SERVICIO`**: cada comentario se parte en **segmentos** (por `. ; ! ?` y conectores adversativos *pero, aunque, sin embargo, eso sí*), y el sentimiento de cada aspecto es el del **segmento donde aparece**. Así un comentario como *"el sonido es genial, pero el ajuste es pésimo"* aporta un aspecto positivo y otro negativo, en vez de una única etiqueta global.

In [ ]:
SEG_RE = re.compile(r"[.;!?¡¿]+|\b(?:pero|aunque|sin embargo|eso sí)\b", re.I)


def split_segments(text):
    spans, start = [], 0
    for m in SEG_RE.finditer(text):
        if text[start:m.start()].strip():
            spans.append((start, m.start()))
        start = m.end()
    if text[start:].strip():
        spans.append((start, len(text)))
    return spans


rows = []
for idx, doc in enumerate(docs_ner):
    spans = split_segments(doc.text)
    for ent in doc.ents:
        if ent.label_ not in ("CARACTERISTICA", "SERVICIO"):
            continue
        k = next((i for i, (s, e) in enumerate(spans) if s <= ent.start_char < e), None)
        if k is not None:
            rows.append((idx, ent.ent_id_, ent.label_, ent.text.lower(), k, doc.text[spans[k][0]:spans[k][1]].strip(" ,:")))
asp = (pd.DataFrame(rows, columns=["row", "aspecto", "tipo", "termino", "seg_id", "segmento"])
       .drop_duplicates(["row", "aspecto", "seg_id"]))

# Sentimiento por segmento único (evita inferencias repetidas)
unique_segments = asp["segmento"].unique().tolist()
seg_sent = dict(zip(unique_segments, predict_sentiment(unique_segments)))
asp["sent_label"] = asp["segmento"].map(lambda s: seg_sent[s][0])
asp["polarity"] = asp["segmento"].map(lambda s: seg_sent[s][1] - seg_sent[s][3])
asp = asp.join(df[["ID", "Fuente", "Producto", "Puntuacion"]], on="row")

by_row = asp.groupby("row")[["aspecto", "sent_label"]].apply(lambda g: list(zip(g["aspecto"], g["sent_label"])))
df["aspectos"] = [by_row.get(i, []) for i in range(len(df))]
print(f"{len(asp)} menciones de aspecto en {asp['row'].nunique()} de {len(df)} comentarios")
display(asp[["ID", "Producto", "aspecto", "segmento", "sent_label", "polarity"]].sample(8, random_state=SEED))

In [ ]:
def aspect_table(a):
    """Tabla POS/NEU/NEG por aspecto + % y sentimiento neto = (POS − NEG) / menciones."""
    t = (a.groupby(["aspecto", "sent_label"]).size().unstack(fill_value=0).reindex(columns=SENT_ORDER, fill_value=0))
    t["menciones"] = t[SENT_ORDER].sum(axis=1)
    t["%POS"] = (t["POS"] / t["menciones"] * 100).round(1)
    t["%NEG"] = (t["NEG"] / t["menciones"] * 100).round(1)
    t["neto"] = ((t["POS"] - t["NEG"]) / t["menciones"]).round(2)
    return t.sort_values("menciones", ascending=False)


display(aspect_table(asp[asp["tipo"] == "CARACTERISTICA"]))

fig, ax = plt.subplots(figsize=(11, 4.5))
tab = (asp[asp["tipo"] == "CARACTERISTICA"].groupby(["aspecto", "sent_label"]).size().unstack(fill_value=0)
       .reindex(columns=SENT_ORDER, fill_value=0))
tab = tab.loc[tab.sum(axis=1).sort_values().index]
tab.plot(kind="barh", stacked=True, color=[SENT_COLORS[c] for c in SENT_ORDER], ax=ax, width=.75)
ax.set_title("Menciones de cada característica y su sentimiento (todos los productos y fuentes)"); ax.set_ylabel("")
ax.legend(title="", frameon=False)
plt.tight_layout(); plt.show()

## Fase 3 — Información para la toma de decisiones y reporte gerencial

### 3.1 Preguntas decisionales (Goldratt)

Goldratt distingue **datos** de **información**: información es *la respuesta a la pregunta que se hace el decisor*. Por eso, antes de graficar nada, fijamos una pregunta concreta por cada desafío estratégico, con **qué se mide, sobre qué datos y para qué decisión**:

| # | Desafío | Pregunta decisional |
|---|---|---|
| **P1** | Rediseño del producto | ¿Cuáles son las **3 características más mencionadas con sentimiento positivo predominante** y las **3 deficiencias más mencionadas con sentimiento negativo predominante** en los auriculares TecnoGadget (reseñas de Amazon, ForoX y X), que deban conservarse o corregirse en el próximo modelo? |
| **P2** | Optimización del servicio | ¿Cuáles son los **3 puntos de dolor más frecuentes** (tiempos de espera, atención, garantía, chatbot...) en las interacciones con el servicio técnico y **cuántos puntos de satisfacción cuestan**, y qué **defectos del producto** originan más contactos, para priorizar mejoras que suban la satisfacción y reduzcan costes? |
| **P3** | Análisis competitivo | ¿En qué **dimensiones** (calidad del producto, funcionalidades, relación calidad-precio) y **aspectos** el sentimiento neto de TecnoGadget está **por encima o por debajo** del mejor de sus dos competidores, y dónde debe actuar para cerrar la brecha? |

In [ ]:
QUESTIONS = {
    "P1": "¿Qué 3 características conservar y qué 3 deficiencias corregir en el rediseño?",
    "P2": "¿Cuáles son los 3 puntos de dolor del servicio técnico y cuánta satisfacción cuestan?",
    "P3": "¿Dónde está TecnoGadget por encima/por debajo de sus competidores?",
}
MIN_MENTIONS = 10          # umbral para no sacar conclusiones de aspectos con muy pocas menciones
CARAC = asp[asp["tipo"] == "CARACTERISTICA"]
RESENAS = CARAC[CARAC["Fuente"] != "ServicioCliente"]      # P1 y P3 usan solo reseñas (el servicio solo atiende a TecnoGadget)

### 3.2 P1 — Rediseño del producto

In [ ]:
q1 = aspect_table(RESENAS[RESENAS["Producto"] == PRODUCT_TG])
q1 = q1[q1["menciones"] >= MIN_MENTIONS]
fortalezas = q1[q1["POS"] > q1["NEG"]].head(3)          # ya ordenado por menciones
deficiencias = q1[q1["NEG"] > q1["POS"]].head(3)
display(q1.style.background_gradient(subset=["neto"], cmap="RdYlGn", vmin=-1, vmax=1))

plot_q1 = q1.sort_values("menciones")
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.barh(plot_q1.index, plot_q1["POS"], color=SENT_COLORS["POS"], label="Positivas")
ax.barh(plot_q1.index, -plot_q1["NEG"], color=SENT_COLORS["NEG"], label="Negativas")
for i, (a, r) in enumerate(plot_q1.iterrows()):
    if a in fortalezas.index or a in deficiencias.index:
        ax.text(0, i, "  ★" if a in fortalezas.index else "  ✖", va="center", ha="left" if a in fortalezas.index else "right", fontsize=11)
ax.axvline(0, color="black", lw=.8)
ax.set_title("TecnoGadget: menciones positivas (→) y negativas (←) por característica   ★ conservar · ✖ corregir")
ax.legend(frameon=False, loc="lower right"); plt.tight_layout(); plt.show()


def fmt(tab):
    return "; ".join(f"**{a}** ({int(r['menciones'])} menciones, {r['%POS']:.0f}% pos. / {r['%NEG']:.0f}% neg.)" for a, r in tab.iterrows())


display(Markdown(f"**Respuesta a P1.** Conservar: {fmt(fortalezas)}.  \nCorregir: {fmt(deficiencias)}."))

### 3.3 P2 — Optimización del servicio al cliente

In [ ]:
svc = df[df["Fuente"] == "ServicioCliente"]
svc_asp = asp[asp["Fuente"] == "ServicioCliente"]

rows = []
for a in SERVICE_ASPECTS:
    ids = set(svc_asp[(svc_asp["aspecto"] == a) & (svc_asp["sent_label"] == "NEG")]["ID"])
    con, sin = svc[svc["ID"].isin(ids)], svc[~svc["ID"].isin(ids)]
    rows.append({"punto de dolor": a, "interacciones con queja": len(ids), "% de interacciones": len(ids) / len(svc) * 100,
                 "satisfacción (con queja)": con["Puntuacion"].mean(), "satisfacción (resto)": sin["Puntuacion"].mean()})
q2 = pd.DataFrame(rows).set_index("punto de dolor")
q2["puntos perdidos"] = q2["satisfacción (resto)"] - q2["satisfacción (con queja)"]
q2 = q2.sort_values("interacciones con queja", ascending=False)
display(q2.round(2))

causas = (svc_asp[(svc_asp["tipo"] == "CARACTERISTICA") & (svc_asp["sent_label"] == "NEG")]
          .groupby("aspecto")["ID"].nunique().sort_values(ascending=False).rename("interacciones"))
causas_pct = (causas / len(svc) * 100).round(1).rename("% de interacciones")
display(pd.concat([causas, causas_pct], axis=1).T)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
q2["interacciones con queja"].sort_values().plot(kind="barh", color="#e76f51", ax=axes[0]); axes[0].set_title("Interacciones con queja por punto de dolor")
q2[["satisfacción (con queja)", "satisfacción (resto)"]].plot(kind="bar", color=["#e76f51", "#2a9d8f"], ax=axes[1], rot=20)
axes[1].set_title("Satisfacción media (1-5): con queja vs. resto"); axes[1].set_xlabel(""); axes[1].legend(frameon=False)
causas.sort_values().plot(kind="barh", color="#264653", ax=axes[2]); axes[2].set_title("Defecto del producto que motiva el contacto")
plt.tight_layout(); plt.show()

top_pain = q2.head(3)
pain_txt = "; ".join(f"**{a}** ({int(r['interacciones con queja'])} interacciones, {r['% de interacciones']:.0f}% del total, −{r['puntos perdidos']:.1f} pts)"
                     for a, r in top_pain.iterrows())
cause_txt = ", ".join(f"**{a}** ({v})" for a, v in causas.head(3).items())
display(Markdown(f"**Respuesta a P2.** Puntos de dolor: {pain_txt}.  \nDefectos que más contactos generan: {cause_txt}. "
                 f"Satisfacción media del servicio: **{svc['Puntuacion'].mean():.2f}/5** "
                 f"({(svc['sentiment_label'] == 'NEG').mean() * 100:.0f}% de interacciones con sentimiento negativo)."))

### 3.4 P3 — Análisis competitivo

In [ ]:
def net_sentiment(s):
    return ((s == "POS").sum() - (s == "NEG").sum()) / len(s)


counts = RESENAS.groupby(["aspecto", "Producto"]).size().unstack(fill_value=0).reindex(columns=PRODUCT_ORDER, fill_value=0)
net = RESENAS.groupby(["aspecto", "Producto"])["sent_label"].agg(net_sentiment).unstack().reindex(columns=PRODUCT_ORDER)
net = net[(counts >= MIN_MENTIONS).all(axis=1)]
net["brecha vs. mejor competidor"] = net[PRODUCT_TG] - net[[PRODUCT_A, PRODUCT_B]].max(axis=1)
net = net.sort_values("brecha vs. mejor competidor", ascending=False)

RESENAS = RESENAS.assign(dimension=RESENAS["aspecto"].map({a: d for d, aa in DIMENSIONES.items() for a in aa}))
dim_net = RESENAS.groupby(["dimension", "Producto"])["sent_label"].agg(net_sentiment).unstack().reindex(columns=PRODUCT_ORDER)
dim_net["brecha vs. mejor competidor"] = dim_net[PRODUCT_TG] - dim_net[[PRODUCT_A, PRODUCT_B]].max(axis=1)
display(dim_net.rename(columns=BRAND_SHORT).round(2))

resumen = (df[df["Fuente"] != "ServicioCliente"].groupby("Producto").agg(
    comentarios=("ID", "size"), puntuacion_media=("Puntuacion", "mean"),
    pct_pos=("sentiment_label", lambda s: (s == "POS").mean() * 100),
    pct_neg=("sentiment_label", lambda s: (s == "NEG").mean() * 100)).round(2).reindex(PRODUCT_ORDER))
display(resumen)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), gridspec_kw={"width_ratios": [1.5, 1]})
sns.heatmap(net[PRODUCT_ORDER].rename(columns=BRAND_SHORT), annot=True, fmt=".2f", cmap="RdYlGn", center=0, vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Sentimiento neto por aspecto y producto (−1 negativo · +1 positivo)"); axes[0].set_ylabel(""); axes[0].set_xlabel("")
gap = net["brecha vs. mejor competidor"].sort_values()
gap.plot(kind="barh", color=np.where(gap >= 0, SENT_COLORS["POS"], SENT_COLORS["NEG"]), ax=axes[1])
axes[1].axvline(0, color="black", lw=.8); axes[1].set_title("Brecha de TecnoGadget vs. el mejor competidor"); axes[1].set_ylabel("")
plt.tight_layout(); plt.show()

TH = 0.10
ventajas, desventajas = net[net["brecha vs. mejor competidor"] >= TH], net[net["brecha vs. mejor competidor"] <= -TH]
fmt_gap = lambda t: ", ".join(f"**{a}** ({r['brecha vs. mejor competidor']:+.2f})" for a, r in t.iterrows()) or "ninguno"
display(Markdown(f"**Respuesta a P3.** TecnoGadget lidera en: {fmt_gap(ventajas)}.  \nVa por detrás en: {fmt_gap(desventajas.sort_values('brecha vs. mejor competidor'))}.  \n"
                 f"Dimensión más débil frente a la competencia: **{dim_net['brecha vs. mejor competidor'].idxmin()}** "
                 f"({dim_net['brecha vs. mejor competidor'].min():+.2f})."))

### 3.5 Prototipo de reporte gerencial

Mini-dashboard de **una sola página** con la respuesta a las tres preguntas, seguido del resumen ejecutivo. Se muestra **solo la información clave** (nada de datos brutos) para evitar el "síndrome del pajar".

In [ ]:
fig = plt.figure(figsize=(20, 10.5))
gs = GridSpec(2, 3, height_ratios=[.11, 1], hspace=.2, wspace=.38)
fig.suptitle("TecnoGadget S.A. — Voz del Cliente: panel para Producto y Marketing", fontsize=18, fontweight="bold", y=.995)

rev_tg = df[(df["Producto"] == PRODUCT_TG) & (df["Fuente"] != "ServicioCliente")]
kpis = [("Comentarios analizados", f"{len(df):,}"),
        ("Satisfacción reseñas TecnoGadget", f"{rev_tg['Puntuacion'].mean():.2f} / 5"),
        ("Reseñas TecnoGadget negativas", f"{(rev_tg['sentiment_label'] == 'NEG').mean() * 100:.0f}%"),
        ("Satisfacción del servicio", f"{svc['Puntuacion'].mean():.2f} / 5")]
kax = fig.add_subplot(gs[0, :]); kax.axis("off")
for i, (label, value) in enumerate(kpis):
    x = .02 + i * .25
    kax.text(x, .45, value, fontsize=26, fontweight="bold", color="#264653", transform=kax.transAxes)
    kax.text(x, .0, label, fontsize=12, color="#555", transform=kax.transAxes)

ax1 = fig.add_subplot(gs[1, 0])
ax1.barh(plot_q1.index, plot_q1["POS"], color=SENT_COLORS["POS"]); ax1.barh(plot_q1.index, -plot_q1["NEG"], color=SENT_COLORS["NEG"])
ax1.axvline(0, color="black", lw=.8); ax1.set_title("P1 · Qué conservar (★) y qué corregir (✖)", fontsize=13, loc="left")
for i, a in enumerate(plot_q1.index):
    if a in fortalezas.index: ax1.text(0, i, "  ★", va="center", fontsize=12)
    if a in deficiencias.index: ax1.text(0, i, "✖  ", va="center", ha="right", fontsize=12)

ax2 = fig.add_subplot(gs[1, 1])
pain = q2.sort_values("interacciones con queja")
ax2.barh(pain.index, pain["% de interacciones"], color="#e76f51")
for i, (a, r) in enumerate(pain.iterrows()):
    ax2.text(r["% de interacciones"] + .5, i, f"−{r['puntos perdidos']:.1f} pts", va="center", fontsize=10)
ax2.set_title("P2 · % de interacciones con queja (y satisfacción perdida)", fontsize=13, loc="left"); ax2.set_xlim(0, pain["% de interacciones"].max() * 1.25)

ax3 = fig.add_subplot(gs[1, 2])
sns.heatmap(net[PRODUCT_ORDER].rename(columns=BRAND_SHORT), annot=True, fmt=".2f", cmap="RdYlGn", center=0, vmin=-1, vmax=1, cbar=False, ax=ax3)
ax3.set_title("P3 · Sentimiento neto vs. competidores", fontsize=13, loc="left"); ax3.set_xlabel(""); ax3.set_ylabel("")
fig.savefig("Dashboard_TecnoGadget.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
ACTIONS = {
    "sonido": "Mantener la firma sonora como eje de la comunicación de marca.",
    "graves": "Mantener; valorar un modo de realce de graves en la app.",
    "cancelación de ruido": "Proteger el ANC como diferencial y reforzarlo en la comunicación frente a la competencia.",
    "batería": "Conservar la autonomía y comunicarla; vigilar la degradación a medio plazo.",
    "comodidad y ajuste": "Rediseñar almohadillas y sujeción (más tallas/aletas) y validar con pruebas ergonómicas de uso prolongado.",
    "conectividad": "Auditar el firmware y el chip Bluetooth, reforzar el QA de emparejamiento y priorizar una actualización OTA.",
    "micrófono": "Mejorar el array de micrófonos y la reducción de viento/eco en la próxima iteración.",
    "app y ecualizador": "Conservar el ecualizador; estabilizar la app (errores y pérdida de perfiles) si hay quejas.",
    "diseño y estuche": "Revisar materiales y bisagra del estuche (resistencia a arañazos y desgaste).",
    "precio": "Revisar posicionamiento de precio o reforzar el valor percibido (accesorios, garantía ampliada).",
    "tiempos de espera": "Fijar un SLA de respuesta, ampliar horario/capacidad y ofrecer devolución de llamada.",
    "garantía y devoluciones": "Simplificar el proceso de garantía: recogida a cargo de la empresa y sustitución anticipada.",
    "atención del agente": "Formación en resolución en primer contacto, guiones menos rígidos y escalado rápido.",
    "chatbot": "Mejorar la derivación a un humano y reducir bucles de menú en el asistente virtual.",
}
act = lambda names: "\n".join(f"- **{a}**: {ACTIONS.get(a, 'Analizar con el equipo responsable.')}" for a in names)

resumen_md = f"""
## Resumen ejecutivo para la gerencia de TecnoGadget S.A.

**Contexto.** Se han analizado {len(df):,} comentarios (reseñas de {', '.join(SOURCE_ORDER[:3])} y transcripciones de servicio) con PLN:
sentimiento, entidades, temas y sentimiento por aspecto.

**1 · Producto (P1).** Los clientes valoran sobre todo {', '.join(f'**{a}**' for a in fortalezas.index)}.
Las deficiencias dominantes son {', '.join(f'**{a}**' for a in deficiencias.index)}.

**2 · Servicio (P2).** Los mayores puntos de dolor son {', '.join(f'**{a}**' for a in top_pain.index)}; los defectos del producto que más contactos
generan son {', '.join(f'**{a}**' for a in causas.head(3).index)}. Corregir el producto reduce también la carga del servicio.

**3 · Competencia (P3).** Ventaja frente al mejor competidor en {', '.join(f'**{a}**' for a in ventajas.index) or 'ningún aspecto'};
desventaja en {', '.join(f'**{a}**' for a in desventajas.index) or 'ningún aspecto'}.

### Recomendaciones accionables
**Conservar y comunicar**
{act(fortalezas.index)}

**Corregir en el próximo modelo**
{act(deficiencias.index)}

**Mejorar el servicio**
{act(top_pain.index)}

**Cerrar brechas competitivas**
{act(desventajas.index.difference(deficiencias.index))}
"""
display(Markdown(resumen_md))

### 3.6 Reflexión crítica sobre la calidad de la información (Goldratt)

- **¿Responde a la pregunta del gerente?** Cada panel contesta una pregunta cerrada (qué conservar/corregir, qué duele más en el servicio, dónde estamos por encima o por debajo). No se entregan listados, sino rankings con umbral mínimo de menciones.
- **Validez del sentimiento**: el modelo es general (no conoce el dominio) y se ha contrastado con las estrellas; el acuerdo no es perfecto porque hay comentarios mixtos y puntuaciones con ruido. El sentimiento por **aspecto** mejora esa limitación al evaluar el segmento concreto.
- **Límites del enfoque por segmentos**: separar por puntuación y conectores no resuelve ironía, referencias cruzadas ni aspectos implícitos (p. ej. *"se me caen al correr"* sin nombrar el ajuste). La taxonomía es manual y habría que mantenerla.
- **NER**: el modelo pequeño de spaCy comete falsos positivos en texto corto y en minúsculas; se mitiga con el `EntityRuler` para el dominio y con una capa de validación, pero la cobertura de personas y lugares sigue siendo orientativa.
- **LDA**: los temas son una ayuda exploratoria; con textos cortos y cercanos a la taxonomía acaban coincidiendo con los aspectos. Su valor está en descubrir temas **no previstos** (en datos reales aparecerían quejas fuera de la taxonomía).
- **Datos simulados**: los resultados **confirman** los perfiles introducidos en la simulación, pero no son evidencia sobre un mercado real. Con datos reales habría que revisar sesgo de muestreo (quien reseña suele estar muy contento o muy enfadado) y la representatividad de cada canal.

### 3.7 Exportación de entregables

In [ ]:
final = df.drop(columns=["raw_tokens", "raw_length"]).copy()
for col in ["processed_tokens", "entidades", "aspectos"]:
    final[col] = final[col].apply(lambda v: json.dumps(v, ensure_ascii=False))
final.to_csv("Datos_TecnoGadget_analizado.csv", index=False, encoding="utf-8")
asp.drop(columns=["row", "seg_id"]).to_csv("Aspectos_TecnoGadget.csv", index=False, encoding="utf-8")

display(final.drop(columns=["texto_limpio", "processed_tokens"]).head(5))
print("Columnas del DataFrame final:", final.columns.tolist())

try:                                    # descarga automática solo en Colab
    from google.colab import files
    for f in ["Datos_TecnoGadget.csv", "Datos_TecnoGadget_analizado.csv", "Dashboard_TecnoGadget.png"]:
        files.download(f)
except ImportError:
    pass